### RESUMO
I. Funções
Blocos de código reutilizáveis que recebem inputs e retornam outputs
Analogia: Como uma receita de bolo - você define os ingredientes (parâmetros) e obtém o resultado

II. Argumentos Posicionais vs Nomeados
Posicionais: Ordem importa. Nomeados: Especifica o parâmetro explicitamente
Analogia: Endereço - "Rua X, número Y" (posicional) vs "número Y da Rua X" (nomeado)

III. Namespaces e Escopo
Variáveis dentro da função são locais, fora são globais
Analogia: Sua casa (local) vs rua (global) - o que acontece na casa fica na casa

IV. Retorno Múltiplo
Na verdade retorna uma tupla que pode ser desempacotada
Analogia: Pacote com vários itens que você desembrulha separadamente

V. Funções como Objetos
Funções podem ser passadas como argumentos, armazenadas em listas
Analogia: Ferramentas em uma caixa - você escolhe qual usar quando precisar

VI. Lambda Functions
Funções anônimas de uma linha, úteis para operações simples
Analogia: Atalho de teclado vs programa completo

VII. Generators
Produzem valores sob demanda, economizando memória
Analogia: Torneira (água quando abre) vs balde cheio (toda água de uma vez)

VIII. Tratamento de Exceções
try/except para lidar com erros graciosamente
Analogia: Airbag - só ativa quando há colisão (erro)

### APLICAÇÃO REAL
I. Engenharia de Dados
Pipelines ETL: Funções para cada etapa (extract, transform, load)
Cloud Functions: Lambdas em AWS/Azure para processamento sob demanda
Validação de Dados: try/except para lidar com dados corrompidos
Processamento Streaming: Generators para dados que chegam continuamente

II. Análise de Dados
Limpeza de Dados: Listas de funções de transformação (como no exemplo dos estados)
Feature Engineering: Lambda functions em operações com DataFrames
Validação: Verificar tipos de dados e formatos antes do processamento

In [2]:
import re
from typing import List, Dict, Union, Generator
from datetime import datetime

# FUNÇÕES DE LIMPEZA (Namespace Global - reutilizáveis)
def remover_caracteres_especiais(texto: str) -> str:
    """Remove caracteres especiais usando regex"""
    return re.sub(r'[!@#$%^&*()]', '', texto)

def normalizar_email(texto: str) -> str:
    """Converte para minúsculas e remove espaços - assume que é email"""
    return texto.strip().lower()

def extrair_e_validar_id(texto: str) -> Union[str, None]:
    """Extrai IDs numéricos de 8 dígitos"""
    try:
        # Encontra sequências de 8 dígitos
        ids = re.findall(r'\d{8}', texto)
        return ids[0] if ids else None
    except (AttributeError, TypeError):
        return None

def classificar_tipo_dado(texto: str) -> str:
    """Classifica o tipo de dado baseado em padrões"""
    texto_limpo = texto.strip().lower()
    
    if re.match(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$', texto_limpo):
        return 'email'
    elif re.match(r'^\d{8}$', texto_limpo):
        return 'id_produto'
    elif any(palavra in texto_limpo for palavra in ['nome', 'joão', 'maria', 'carlos']):
        return 'nome_usuario'
    else:
        return 'desconhecido'

# LISTA DE OPERAÇÕES GLOBAL (reutilizável)
OPS_LIMPEZA_GERAIS = [
    str.strip,
    remover_caracteres_especiais,
    lambda x: normalizar_email(x) if '@' in x else x,
    lambda x: x.title() if any(nome in x.lower() for nome in ['joão', 'maria', 'carlos']) else x
]

# PIPELINE PRINCIPAL CORRIGIDA
def pipeline_limpeza_dados(dados_brutos: List[str], 
                          operacoes: List[callable] = None) -> List[Dict]:
    """
    Pipeline robusto para limpar dados de usuários de e-commerce
    """
    
    # Usa operações padrão se nenhuma for fornecida
    if operacoes is None:
        operacoes = OPS_LIMPEZA_GERAIS
    
    # Generator CORRETO com tipo definido
    def processar_lote(dados: List[str]) -> Generator[str, None, None]:
        """Generator para processamento lazy com tratamento de erro"""
        for linha in dados:
            try:
                if not linha or not isinstance(linha, str):
                    continue
                    
                # Aplica cadeia de operações
                dado_processado = linha
                for operacao in operacoes:
                    dado_processado = operacao(dado_processado)
                
                yield dado_processado
                
            except Exception as e:
                print(f"🚨 Erro processando '{linha}': {e}")
                continue  # Continua com próximo item
    
    # Processamento principal
    dados_processados = []
    for dado_limpo in processar_lote(dados_brutos):
        if dado_limpo and dado_limpo.strip():  # Filtra vazios
            dados_processados.append({
                'dado_original': next((d for d in dados_brutos if d.strip() == dado_limpo), 'N/A'),
                'dado_limpo': dado_limpo,
                'tipo': classificar_tipo_dado(dado_limpo),
                'timestamp': datetime.now().isoformat(),
                'comprimento_original': len(dado_limpo)
            })
    
    return dados_processados

# USO PRÁTICO CORRIGIDO
def demonstrar_pipeline():
    """Demonstra o funcionamento do pipeline"""
    
    dados_sujos = [
        "  JOÃO@EMAIL.COM  ",
        "maria!silva#gmail.com",
        "produto123",
        "  CARLOS SANTOS  ",
        "   ",  # string vazia
        "ID: 12345678",  # ID válido
        "2024-01-15"  # data
    ]
    
    print("=== PIPELINE DE LIMPEZA DE DADOS ===")
    print(f"📥 Input: {len(dados_sujos)} registros")
    
    # Método 1: Usando operações padrão
    resultado1 = pipeline_limpeza_dados(dados_sujos)
    print(f"📤 Output (padrão): {len(resultado1)} registros limpos")
    
    # Método 2: Operações customizadas
    ops_customizadas = [
        str.strip,
        lambda x: x.upper(),  # Força maiúsculas
        remover_caracteres_especiais
    ]
    
    resultado2 = pipeline_limpeza_dados(dados_sujos, ops_customizadas)
    print(f"📤 Output (customizado): {len(resultado2)} registros limpos")
    
    # Demonstrando resultados
    print("\n🔍 DETALHES DOS RESULTADOS:")
    for i, item in enumerate(resultado1[:3]):  # Mostra apenas 3 primeiros
        print(f"  {i+1}. Original: '{item['dado_original']}'")
        print(f"     Limpo: '{item['dado_limpo']}'")
        print(f"     Tipo: {item['tipo']}")
        print()

# EXECUÇÃO SEGURA COM TRY/EXCEPT
if __name__ == "__main__":
    try:
        demonstrar_pipeline()
        
        # Teste adicional com dados problemáticos
        print("=== TESTE COM DADOS PROBLEMÁTICOS ===")
        dados_problematicos = [
            None,  # None value
            123,   # Número
            "",    # String vazia
            "  TESTE@EMAIL.COM  "
        ]
        
        resultado_seguro = pipeline_limpeza_dados([str(x) if x else "" for x in dados_problematicos])
        print(f"✅ Processados com segurança: {len(resultado_seguro)} de {len(dados_problematicos)}")
        
    except Exception as e:
        print(f"❌ Erro na execução: {e}")

=== PIPELINE DE LIMPEZA DE DADOS ===
📥 Input: 7 registros
📤 Output (padrão): 6 registros limpos
📤 Output (customizado): 6 registros limpos

🔍 DETALHES DOS RESULTADOS:
  1. Original: 'N/A'
     Limpo: 'Joãoemail.Com'
     Tipo: nome_usuario

  2. Original: 'N/A'
     Limpo: 'Mariasilvagmail.Com'
     Tipo: nome_usuario

  3. Original: 'produto123'
     Limpo: 'produto123'
     Tipo: desconhecido

=== TESTE COM DADOS PROBLEMÁTICOS ===
✅ Processados com segurança: 2 de 4
